# The Time Diversification Effect

In [ ]:
import pandas as pd
import numpy as np
import random
from matplotlib import pyplot as plt
import seaborn as sns
from datetime import date
from dateutil.relativedelta import relativedelta

from trading_algos import optimization as tao
from trading_algos import datasets as tad
from trading_algos import plots as tap
from trading_algos import utils as tau
from trading_algos.utils import head_tail as ht

%load_ext autoreload
%autoreload 2

## Load Data

In [ ]:
# Load a complete collection of S&P500 stocks from repository
# Not interested in survivors here
df_stocks = tad.get_sp500(survivors=False)

In [ ]:
ht(df_stocks)

In [ ]:
df_stocks.shape

In [ ]:
print(df_stocks.columns.get_level_values(1).nunique(), 'stocks')

## Simulating Portfolios

In [ ]:
random.seed(42)

# The number of stocks/start-date combinations to simulate expanding horizon for
n_portfolios = 20

# The number of months that each portfolio will be iterated over
i_months = 60

# Initialize lists that will form the final dataset
ticker_list = []
dt_start = []
dt_end = []
i_months_held = []
inv_risk = []
inv_return = []
inv_sharpe = []

# Keep a list of potentially corrupted Tickers to remove
corrupted_tickers = []

# The date that df_stocks runs until
stocks_end_date = df_stocks.index[-1]
# Assuming each month is a 21 day trading period to ensure all months
# are treated as statistically equal
latest_start_date_idx = (len(df_stocks) - 1) - i_months*21
lastest_start_date = df_stocks.index[(len(df_stocks) - 1) - i_months*21]
date_selection = df_stocks.loc[:lastest_start_date].index.tolist()

# Number of assets in the portfolio
for n in range(1, n_portfolios+1):
    
    # Randomize a starting investment date so that all market
    # conditions are covered
    start_date = random.choice(date_selection)
    start_date_idx = date_selection.index(start_date)

    # Select from a list of stocks that existed across the full 20 years
    latest_end_date_idx = start_date_idx + i_months*21
    latest_end_date = df_stocks.iloc[latest_end_date_idx].name

    df_sim_stocks = df_stocks.loc[start_date:latest_end_date].dropna(axis=1).copy()

    # Select a stock at random that will be invested in
    avail_tickers = df_sim_stocks.columns.get_level_values(1).unique().tolist()
    ticker = random.choice(avail_tickers)

    # Number of randomly allocated n sized portfolios to generate
    # Starting at 3 month investment to reduce excess noise at the start
    for i in range(3, i_months+1, 3):

        end_date_idx = start_date_idx + i*21
        end_date = df_stocks.iloc[end_date_idx].name

        # Calculate equal weighted portfolio returns
        df_returns = tao.calc_returns(df_sim_stocks.loc[start_date:end_date, pd.IndexSlice[:, ticker]])

        # Calculate summary statistics for the portfolio
        df_summary = tau.get_returns_summary(df_returns)

        # After some digging I have found that some stocks have corrupted
        # data due to stock splits, bankrupcies etc. Let's do our best to 
        # Ignore them
        if i > 5 and abs(df_summary.loc[ticker, 'Risk']) > 1:
            if ticker not in corrupted_tickers:
                print(f'Skipping Ticker {ticker} due to potentially corruped data')
                corrupted_tickers.append(ticker)
            continue

        # Store simulation values
        ticker_list.append(ticker)
        dt_start.append(start_date)
        dt_end.append(end_date)
        i_months_held.append(i)
        inv_risk.append(df_summary.loc[ticker, 'Risk'])
        inv_return.append(df_summary.loc[ticker, 'Return'])
        inv_sharpe.append(df_summary.loc[ticker, 'Sharpe'])
        
df_diversification = pd.DataFrame({'Ticker': ticker_list,
                                   'Start' : dt_start,
                                   'End': dt_end,
                                   'Months Held': i_months_held,
                                   'Ann Risk': inv_risk, 
                                   'Ann Return':inv_return,
                                   'Ann Sharpe': inv_sharpe})

df_diversification['Inv Return'] = (1 + df_diversification['Ann Return'])**(df_diversification['Months Held'] / 12) - 1
df_diversification['Inv Risk'] = df_diversification['Ann Risk']*np.sqrt(df_diversification['Months Held'] / 12)
df_diversification['Inv Sharpe'] = df_diversification['Inv Return'] / df_diversification['Inv Risk']
df_diversification['Inv Profit'] = (df_diversification['Inv Return'] > 0) * 1

In [ ]:
df_diversification2 = df_diversification[df_diversification['Ticker'].isin(corrupted_tickers) == False].copy().reset_index(drop=True)
ht(df_diversification)

In [ ]:
df_diversification2.groupby('Months Held').agg({'Inv Profit':np.mean}).plot()

In [ ]:
df_diversification2.groupby('Months Held').agg({'Ann Return':np.std}).plot()

In [ ]:
df_diversification2.groupby('Months Held').agg({'Ann Risk':np.mean}).plot()

In [ ]:
df_diversification.groupby('Months Held').agg({'Inv Risk':np.mean}).plot()

In [ ]:
df_diversification.iloc[ht(df_diversification[df_diversification['Months Held']>5]['Ann Risk Delta'].sort_values(),15).index]

In [ ]:
df_diversification.iloc[df_diversification['Ann Risk'].sort_values(ascending=False).head(100).index]#['Ticker'].unique()

In [ ]:
tao.calc_returns(df_stocks.loc[:end_date, pd.IndexSlice[:, 'NVR']])

In [ ]:
df_diversification.iloc[df_diversification['Return'].idxmax()]

In [ ]:
df_diversification.groupby('Years Held').agg({'Return':np.max})

In [ ]:
df_diversification.groupby('Years Held').agg({'Return':np.std}).plot()

In [ ]:
# 5 rows, 4 columns
r,c = 5,4
fig, ax = plt.subplots(r, c, figsize=(12, 12), tight_layout=True, sharex=True)

fig.tight_layout(pad=2, rect=[0,0.05,1,0.95])
fig.suptitle(f'Distribution of Portfolio Sharpe Ratios Narrows as Asset Count Increases',
             fontsize=16)
fig.supxlabel('Sharpe Ratio', y=0.05, fontsize=12)
fig.supylabel('Frequency', fontsize=12)

for n in df_diversification['Years Held'].unique():

    # Axes row
    ax_r = int(np.floor((n-1)/c))
    # Axes column
    ax_c = int((n-1)%c)
    # Axes subplot
    ax_p = ax[ax_r, ax_c]

    ax_p.set_title(f'n = {n}', fontsize=12, fontstyle='italic')
    ax_p.spines[['left','top', 'right']].set_visible(False)
    ax_p.set_yticklabels([])
    ax_p.set_yticks([])
    ax_p.set_ylabel(' ')
    ax_p.set_xlabel(' ')

    sns.histplot(
        df_diversification[df_diversification['Years Held']==n]['Sharpe'], 
        ax=ax_p,
        alpha=0.1,
        kde=True,
        bins=10
        )

footnote = 'Random portfolios converge toward a stable Sharpe distribution as asset count rises.'

fig.text(0.5,
         0.015,
         footnote,
         ha='center',
         wrap=True,
         fontsize=10)

In [ ]:
df_grouped = df_diversification.groupby('Years Held').mean()[['Return', 'Risk', 'Sharpe']]
df_grouped.index = df_grouped.index.astype(str)
df_grouped

In [ ]:
df_grouped['Risk'].plot()

In [ ]:
tap.plot_summary(df_grouped);

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(12, 6), tight_layout=True, sharex=True)

fig.suptitle('Increasing Assets Reduces Risk, Stabilizes Returns and Improves Performance', fontsize=16)
fig.supxlabel('N Assets')

ax[0].set_ylabel('ann. Risk',fontsize=12)
ax[0].set_xticks(np.arange(0,len(df_grouped)))
ax[0].spines[['top', 'right']].set_visible(False)
ax[0].plot(df_grouped['Risk'])

ax[1].set_ylabel('ann. Ret',fontsize=12)
ax[1].set_xticks(np.arange(0,len(df_grouped)))
ax[1].spines[['top', 'right']].set_visible(False)
ax[1].plot(df_grouped['Return'])

ax[2].set_ylabel('Sharpe',fontsize=12)
ax[2].set_xticks(np.arange(0,len(df_grouped)))
ax[2].spines[['top', 'right']].set_visible(False)
ax[2].plot(df_grouped['Sharpe'])

plt.show();